# Blood Donor Eligibility — Random Forest (FIXED)

### Root cause of the previous 100% scores
The cleaned dataset contained `has_medical_condition`, `low_hemoglobin`, and `low_weight` — three binary flags that **exactly reconstruct the target label** (100% rule match confirmed). Removing them and training on raw features produces realistic, meaningful scores.

### Expected realistic scores (after fix)
- Test F1 ≈ 0.87–0.90 (Random Forest is well-suited here)
- Test AUC ≈ 0.75–0.80
- Train-Test gap < 0.05 (no overfitting)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

In [2]:
# ============================================================
# 1. LOAD & PREPARE DATA FROM RAW SOURCE
# ============================================================

candidate_paths = [
    Path("blood_donation.csv"),
    Path("../DATASET/blood_donation.csv"),
]
dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find blood_donation.csv")

df = pd.read_csv(dataset_path)
print(f"Loaded: {dataset_path}")
print("Raw shape:", df.shape)

FileNotFoundError: Could not find blood_donation.csv

In [ ]:
# ============================================================
# 2. FEATURE ENGINEERING (Leakage-Free)
# ============================================================

REFERENCE_DATE = pd.Timestamp("2025-01-01")

df["Last_Donation_Date"] = df["Last_Donation_Date"].replace("Never", np.nan)
df["Last_Donation_Date"] = pd.to_datetime(df["Last_Donation_Date"], dayfirst=True, errors="coerce")
df["Registration_Date"] = pd.to_datetime(df["Registration_Date"], dayfirst=True, errors="coerce")

df["days_since_last_donation"] = (REFERENCE_DATE - df["Last_Donation_Date"]).dt.days
df["days_since_last_donation"] = df["days_since_last_donation"].fillna(
    (REFERENCE_DATE - df["Registration_Date"]).dt.days
)

df["required_interval"] = df["Gender"].apply(lambda x: 90 if str(x).lower() == "male" else 120)
df["eligible_by_interval"] = (df["days_since_last_donation"] >= df["required_interval"]).astype(int)

df["Gender_Male"]   = (df["Gender"].str.lower() == "male").astype(int)
df["Gender_Female"] = (df["Gender"].str.lower() == "female").astype(int)

df["Eligible_for_Donation_binary"] = df["Eligible_for_Donation"].map({"Yes": 1, "No": 0})

# LEAKY FEATURES REMOVED:
# - has_medical_condition (100% determines eligibility for medical cases)
# - low_hemoglobin        (threshold-derived, directly encodes ineligibility rule)
# - low_weight            (threshold-derived, directly encodes ineligibility rule)

FEATURES = [
    "Hemoglobin_g_dL",
    "Weight_kg",
    "Age",
    "days_since_last_donation",
    "eligible_by_interval",
    "Total_Donations",
    "Gender_Male",
    "Gender_Female",
]

X = df[FEATURES]
y = df["Eligible_for_Donation_binary"]

print("Feature set shape:", X.shape)
print("\nTarget distribution (%):\n",
      (y.value_counts(normalize=True) * 100).round(2))

In [ ]:
# ============================================================
# 3. STRATIFIED TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training size:", X_train.shape)
print("Testing size: ", X_test.shape)

In [ ]:
# ============================================================
# 4. HYPERPARAMETER SEARCH WITH REGULARIZATION CONTROLS
# ============================================================

base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid = {
    "n_estimators":     [200, 400],
    "max_depth":        [6, 8, 12],
    "min_samples_split":[10, 20],
    "min_samples_leaf": [5, 10],
    "max_features":     ["sqrt", 0.5],
    "class_weight":     ["balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=base_rf,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

search.fit(X_train, y_train)
best_rf = search.best_estimator_

print("\nBest Parameters:")
print(search.best_params_)
print(f"Best CV F1: {search.best_score_:.4f}")

In [ ]:
# ============================================================
# 5. EVALUATE — TRAIN VS TEST (OVERFITTING CHECK)
# ============================================================

train_pred  = best_rf.predict(X_train)
test_pred   = best_rf.predict(X_test)
train_proba = best_rf.predict_proba(X_train)[:, 1]
test_proba  = best_rf.predict_proba(X_test)[:, 1]

train_f1  = f1_score(y_train, train_pred)
test_f1   = f1_score(y_test, test_pred)
train_auc = roc_auc_score(y_train, train_proba)
test_auc  = roc_auc_score(y_test, test_proba)
overfit_gap = train_f1 - test_f1

print(f"Train Accuracy: {accuracy_score(y_train, train_pred):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, test_pred):.4f}")
print(f"Train F1:       {train_f1:.4f}")
print(f"Test F1:        {test_f1:.4f}")
print(f"Train ROC-AUC:  {train_auc:.4f}")
print(f"Test ROC-AUC:   {test_auc:.4f}")
print(f"Overfitting Gap: {overfit_gap:.4f}")

if overfit_gap > 0.05:
    print("⚠️  Warning: Noticeable overfitting. Tighten depth/leaf constraints.")
else:
    print("✅ Good: Overfitting gap is acceptable.")

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred, digits=4))

In [ ]:
# ============================================================
# 6. FEATURE IMPORTANCE
# ============================================================

feature_importance = (
    pd.Series(best_rf.feature_importances_, index=FEATURES)
    .sort_values(ascending=False)
)

print("Feature Importance:")
print(feature_importance.round(4))